<a href="https://colab.research.google.com/github/CristianoMSilva/DistilBERT-FL-Playground/blob/main/02_tokenization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Load and clean the dataset (same steps as 01_eda.ipynb)
from datasets import load_dataset
import pandas as pd

df = load_dataset("zefang-liu/phishing-email-dataset")["train"].to_pandas()
df = df.drop(columns=["Unnamed: 0"])
df = df.dropna(subset=["Email Text"])

# Remove texts with conflicting labels, then duplicates
conflicts = df.groupby("Email Text")["Email Type"].nunique()
df = df[~df["Email Text"].isin(conflicts[conflicts > 1].index)]
df = df.drop_duplicates(subset="Email Text")

# Remove empty and corrupted texts
word_count = df["Email Text"].str.split().str.len()
df = df[(word_count > 0) & (word_count < 1_000_000)].reset_index(drop=True)

print("Rows:", len(df))
print(df["Email Type"].value_counts())

README.md:   0%|          | 0.00/616 [00:00<?, ?B/s]

Phishing_Email.csv: reconstructing file:   0%|          |  0.00B / 52.0MB            

Phishing_Email.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/18650 [00:00<?, ? examples/s]

Rows: 17534
Email Type
Safe Email        10978
Phishing Email     6556
Name: count, dtype: int64


In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
print("Vocabulary size:", tokenizer.vocab_size)

#Example
sample = "URGENT: verify your PayPal account now to avoid suspension!"
encoded = tokenizer(sample, padding="max_length", max_length=24, truncation=True)

print("\nTokens:", tokenizer.convert_ids_to_tokens(encoded["input_ids"]))
print("\nIDs:", encoded["input_ids"])
print("\nAttention mask:", encoded["attention_mask"])

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Vocabulary size: 30522

Tokens: ['[CLS]', 'urgent', ':', 'verify', 'your', 'pay', '##pal', 'account', 'now', 'to', 'avoid', 'suspension', '!', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']

IDs: [101, 13661, 1024, 20410, 2115, 3477, 12952, 4070, 2085, 2000, 4468, 8636, 999, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [4]:
encodings = tokenizer(df["Email Text"].tolist(), truncation=False)
df["token_count"] = [len(ids) for ids in encodings["input_ids"]]

print("Median tokens per class:")
print(df.groupby("Email Type")["token_count"].median())

print()
for limit in [128, 256, 512]:
    share = (df["token_count"] > limit).mean()
    print(f"Emails longer than {limit} tokens: {share:.1%}")


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (616 > 512). Running this sequence through the model will result in indexing errors


Median tokens per class:
Email Type
Phishing Email    202.0
Safe Email        277.0
Name: token_count, dtype: float64

Emails longer than 128 tokens: 71.5%
Emails longer than 256 tokens: 48.0%
Emails longer than 512 tokens: 23.9%


In [5]:
from sklearn.model_selection import train_test_split

# Labels as numbers: 1 = phishing, 0 = safe
df["label"] = (df["Email Type"] == "Phishing Email").astype(int)

# 80% train, 10% validation, 10% test, keeping the class proportion in each part
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["label"], random_state=42)

for name, part in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"{name}: {len(part)} emails, {part['label'].mean():.1%} phishing")

Train: 14027 emails, 37.4% phishing
Validation: 1753 emails, 37.4% phishing
Test: 1754 emails, 37.4% phishing
